# Sen1Floods11 Native Segmentation BWER Audit

This notebook is the convenient Colab path for the paper-grade Sen1Floods11 native pixel-level segmentation audit. It does one 64-chip validation run first, checks the required outputs, then runs all selected hand-labeled chips if validation passes.

Recommended Colab runtime: L4 GPU + High RAM. Public Sen1Floods11 documentation reports 446 hand-labeled chips; `OFFICIAL_MAX_SAMPLES = 0` below means prepare all selected candidates, so this should run the full hand-labeled set when the official bucket listing resolves cleanly.

In [ ]:
# === One clear config cell ===
from pathlib import Path

# GitHub repo to clone. Change this after uploading your fork/repo.
REPO_URL = "https://github.com/strivekboy-coder/rsfm-fairness-audit.git"
BRANCH = None  # None clones the repo default branch; set "master" or "main" only when you need a fixed branch.

# Persistent Drive cache. Prepared datasets and zipped outputs are saved here.
DRIVE_ROOT = Path("/content/drive/MyDrive/rsfm_fairness_audit")

# At most one validation run, then official run if validation passes.
RUN_VALIDATION_FIRST = True
RUN_OFFICIAL_AFTER_VALIDATION = True

# Validation should be small and fast.
VALIDATION_MAX_SAMPLES = 64
VALIDATION_CANDIDATE_LIMIT = 1000

# Official setting. Public hand-labeled Sen1Floods11 is 446 chips; 0 means all selected candidates.
OFFICIAL_MAX_SAMPLES = 0
OFFICIAL_CANDIDATE_LIMIT = 100000

# Optional event filters, e.g. ["India", "Pakistan"]. Keep empty for the default manifest order.
EVENT_FILTERS = []

# Prithvi/TerraTorch may need network downloads on first run.
ALLOW_HF_DOWNLOAD = True

PROJECT_DIR = Path("/content/rsfm-fairness-audit")
CONTENT_DATA = Path("/content/data")
CONTENT_OUTPUTS = Path("/content/outputs")

In [ ]:
# Mount Drive and clone/update the repo.
from google.colab import drive
import os, shutil, subprocess, textwrap

drive.mount('/content/drive')
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
(DRIVE_ROOT / 'prepared_zips').mkdir(exist_ok=True)
(DRIVE_ROOT / 'outputs').mkdir(exist_ok=True)
(DRIVE_ROOT / 'cache').mkdir(exist_ok=True)
CONTENT_DATA.mkdir(parents=True, exist_ok=True)
CONTENT_OUTPUTS.mkdir(parents=True, exist_ok=True)

def run(cmd, cwd=None):
    print('\n$ ' + ' '.join(map(str, cmd)))
    subprocess.run([str(x) for x in cmd], cwd=cwd, check=True)

git_dir = PROJECT_DIR / '.git'
if PROJECT_DIR.exists() and git_dir.exists():
    # Existing good checkout: update the currently checked-out/default branch.
    run(['git', 'pull', '--ff-only'], cwd=PROJECT_DIR)
    if BRANCH:
        run(['git', 'fetch', '--all'], cwd=PROJECT_DIR)
        run(['git', 'checkout', BRANCH], cwd=PROJECT_DIR)
        run(['git', 'pull', '--ff-only'], cwd=PROJECT_DIR)
else:
    # Colab can leave a non-empty partial clone after a failed run. This is only /content code, never Drive/cache data.
    if PROJECT_DIR.exists():
        print('Removing stale non-git project directory:', PROJECT_DIR)
        shutil.rmtree(PROJECT_DIR)
    clone_cmd = ['git', 'clone', REPO_URL, str(PROJECT_DIR)]
    if BRANCH:
        clone_cmd = ['git', 'clone', '--branch', BRANCH, REPO_URL, str(PROJECT_DIR)]
    run(clone_cmd)

os.chdir(PROJECT_DIR)
print('Repo ready:', PROJECT_DIR)


In [ ]:
# Runtime sanity check. GPU helps Prithvi inference; data prep and BWER are mostly CPU/IO.
import torch, psutil
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
print('RAM GB:', round(psutil.virtual_memory().total / 1e9, 1))
if not torch.cuda.is_available():
    print('WARNING: CPU runtime is OK for setup/data prep, but official Prithvi segmentation inference will be slow.')

In [ ]:
# Install dependencies. If Colab reports "numpy.dtype size changed" after this, restart runtime once and resume from this notebook.
run(['python', '-m', 'pip', 'install', '-q', '-e', '.'], cwd=PROJECT_DIR)
run(['python', '-m', 'pip', 'install', '-q', '-r', 'requirements-prithvi.txt'], cwd=PROJECT_DIR)

# Persist the honest protocol label in the local config for this run.
cfg = PROJECT_DIR / 'configs/models/prithvi.yaml'
text = cfg.read_text()
if 'allow_hf_download:' in text:
    text = '\n'.join('allow_hf_download: true' if line.startswith('allow_hf_download:') else line for line in text.splitlines()) + '\n'
elif ALLOW_HF_DOWNLOAD:
    text += '\nallow_hf_download: true\n'
cfg.write_text(text)
print(cfg.read_text())

In [ ]:
# Helpers: prepare data in /content, but cache zip files in Drive.
import json, math, zipfile
import pandas as pd

def zip_dir(src: Path, dst: Path):
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.exists():
        dst.unlink()
    with zipfile.ZipFile(dst, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
        for path in src.rglob('*'):
            if path.is_file():
                zf.write(path, path.relative_to(src))

def unzip_to(zip_path: Path, dst: Path):
    if dst.exists():
        shutil.rmtree(dst)
    dst.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(dst)

def prepare_dataset(name: str, max_samples: int, candidate_limit: int) -> Path:
    data_dir = CONTENT_DATA / name
    zip_path = DRIVE_ROOT / 'prepared_zips' / f'{name}.zip'
    if zip_path.exists():
        print('Using cached prepared dataset:', zip_path)
        unzip_to(zip_path, data_dir)
        meta = pd.read_csv(data_dir / 'metadata.csv')
        print(f'Prepared metadata: {len(meta)} chips, {meta.get("event", meta.get("event_id")).nunique()} events')
        return data_dir
    cmd = [
        'python', 'scripts/prepare_sen1floods11_subset.py',
        '--output-dir', str(data_dir),
        '--cache-dir', str(DRIVE_ROOT / 'cache' / 'sen1floods11'),
        '--max-samples', str(max_samples),
        '--candidate-limit', str(candidate_limit),
    ]
    for event in EVENT_FILTERS:
        cmd += ['--event-filter', event]
    run(cmd, cwd=PROJECT_DIR)
    meta = pd.read_csv(data_dir / 'metadata.csv')
    print(f'Prepared metadata: {len(meta)} chips, {meta.get("event", meta.get("event_id")).nunique()} events')
    zip_dir(data_dir, zip_path)
    print('Cached prepared dataset:', zip_path)
    return data_dir

def run_segmentation_audit(label: str, data_dir: Path) -> Path:
    out = CONTENT_OUTPUTS / f'prithvi_sen1floods11_{label}'
    if out.exists():
        shutil.rmtree(out)
    cmd = [
        'python', '-m', 'rsfm_fairness_audit.cli', 'run-segmentation-real',
        '--dataset', 'sen1floods11',
        '--model', 'prithvi',
        '--data-root', str(data_dir),
        '--model-config', 'configs/models/prithvi.yaml',
        '--output-dir', str(out),
    ]
    run(cmd, cwd=PROJECT_DIR)
    zip_dir(out, DRIVE_ROOT / 'outputs' / f'{out.name}.zip')
    return out

def validate_outputs(out: Path) -> bool:
    required = ['event_segmentation_metrics.csv', 'segmentation_audit_table.csv', 'slice_support_recommendations.csv', 'warnings.json', 'bwer_summary.csv', 'bwer_by_slice.csv', 'report.md', 'segmentation_metrics.csv', 'diagnostic_baseline_comparison.csv']
    missing = [name for name in required if not (out / name).exists()]
    if missing:
        raise RuntimeError(f'Missing required outputs: {missing}')
    event = pd.read_csv(out / 'event_segmentation_metrics.csv')
    audit = pd.read_csv(out / 'segmentation_audit_table.csv')
    summary = pd.read_csv(out / 'bwer_summary.csv')
    chip_metrics = pd.read_csv(out / 'segmentation_metrics.csv')
    if event.empty or audit.empty or summary.empty or chip_metrics.empty:
        raise RuntimeError('Validation failed: one or more key tables are empty.')
    if summary['bwer'].isna().all():
        raise RuntimeError('Validation failed: bwer_summary.csv has all-NaN BWER.')
    print('Validation passed.')
    display(summary.head())
    display(event.head())
    diagnostic_cols = [
        'sample_id', 'event_id', 'iou', 'dice',
        'predicted_positive_pixel_ratio', 'ground_truth_positive_pixel_ratio',
        'valid_pixel_count', 'label_values_distribution', 'prediction_unique_values',
        'input_band_order', 'input_raw_min', 'input_raw_max',
        'input_normalized_min', 'input_normalized_max', 'mask_resize_alignment',
    ]
    available = [col for col in diagnostic_cols if col in chip_metrics.columns]
    print('Chip-level diagnostic preview:')
    display(chip_metrics[available].head(10))
    if 'predicted_positive_pixel_ratio' in chip_metrics.columns:
        pred_ratio = chip_metrics['predicted_positive_pixel_ratio'].astype(float)
        extreme = chip_metrics[(pred_ratio <= 0.01) | (pred_ratio >= 0.99)]
        print('Predicted positive ratio summary:')
        display(pred_ratio.describe())
        if len(extreme):
            print(f'WARNING: {len(extreme)} chips have predicted_positive_pixel_ratio <= 0.01 or >= 0.99. Inspect threshold/head/label mapping before interpreting model quality.')
            display(extreme[available].head(20))
    if 'ground_truth_positive_pixel_ratio' in chip_metrics.columns:
        print('Ground-truth positive ratio summary:')
        display(chip_metrics['ground_truth_positive_pixel_ratio'].astype(float).describe())
    baseline = pd.read_csv(out / 'diagnostic_baseline_comparison.csv')
    print('Diagnostic baseline comparison, overall first:')
    display(baseline.assign(_overall=(baseline['event_id'] == '__overall__')).sort_values(['_overall', 'micro_iou'], ascending=[False, False]).drop(columns=['_overall']).head(20))
    print((out / 'warnings.json').read_text())
    return True


In [ ]:
# One validation run. This should be the only small dry run before official results.
VALIDATION_OK = not RUN_VALIDATION_FIRST
if RUN_VALIDATION_FIRST:
    validation_data = prepare_dataset('sen1floods11_validation64', VALIDATION_MAX_SAMPLES, VALIDATION_CANDIDATE_LIMIT)
    validation_out = run_segmentation_audit('validation64', validation_data)
    VALIDATION_OK = validate_outputs(validation_out)
VALIDATION_OK

In [ ]:
# Official run. With OFFICIAL_MAX_SAMPLES = 0, this prepares/runs all selected hand-labeled candidates.
if RUN_OFFICIAL_AFTER_VALIDATION:
    if not VALIDATION_OK:
        raise RuntimeError('Validation did not pass; refusing to run official audit.')
    official_name = f'sen1floods11_official_{OFFICIAL_MAX_SAMPLES or "all"}'
    official_data = prepare_dataset(official_name, OFFICIAL_MAX_SAMPLES, OFFICIAL_CANDIDATE_LIMIT)
    official_out = run_segmentation_audit(f'official_{OFFICIAL_MAX_SAMPLES or "all"}', official_data)
    validate_outputs(official_out)
    print('Official output dir:', official_out)
    print('Drive zip:', DRIVE_ROOT / 'outputs' / f'{official_out.name}.zip')

## What to inspect first

Open these files in the official output directory first:

- `event_segmentation_metrics.csv`: event-level TP/FP/FN/TN, valid pixels, IoU/Dice/F1/precision/recall.
- `slice_support_recommendations.csv`: whether formal BWER variants are recommended or invalid.
- `warnings.json`: invalid balance variables or support warnings.
- `bwer_summary.csv`: raw event-level BWER.
- `report.md`: compact narrative report.

Interpret `event_id` as an operational disaster-event slice, not a causal country fairness attribute.